# 04_204 · Transformer jerárquico multitarea con cuatro categorías

Variante reproducible de `04_5`. Un solo encoder comparte una cabeza binaria `SEGURO`/daño y una cabeza multietiqueta con `RACISMO_DISCRIMINACION`, `ACOSO_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. El score operativo es `p(daño) × p(categoría | representación)`.

`ACOSO_AMENAZA` fusiona acoso personal y amenaza directa. Por defecto se inicia desde el Transformer de `04_2` seleccionado únicamente con validation; todas las cabezas se vuelven a optimizar y los umbrales se recalibran.

El arranque híbrido usa el workspace en local y, en Colab, código fijado de GitHub con artefactos mínimos persistentes en Google Drive.

In [ ]:
from pathlib import Path
from importlib.util import find_spec
import json, os, shutil, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

REPO_URL = 'https://github.com/lkoc/Trabajo_PLN-MIA-Grupo4.git'
GIT_COMMIT = '131016751517333284ee24559d51d2bdebebc960'
PROJECT_NAME, DRIVE_BUNDLE_NAME, NEEDS_PEFT = 'Trabajo_PLN-MIA-Grupo4', 'PLN_colab_04_artifacts', False

def _bootstrap_04_20x():
    in_colab = find_spec('google.colab') is not None
    if not in_colab:
        start = Path.cwd().resolve()
        root = next((p for p in (start, *start.parents) if (p / 'scripts_auxiliares').is_dir()), None)
        if root is None: raise FileNotFoundError('No se encontró la raíz local del proyecto.')
        return root, False, root, 'working-tree-local'
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    artifacts = Path('/content/drive/MyDrive') / DRIVE_BUNDLE_NAME
    manifest_path = artifacts / 'MANIFIESTO_ARTEFACTOS_04_20X.json'
    if not manifest_path.is_file(): raise FileNotFoundError('Falta el bundle de Drive; ejecute sincronizar_04_20x_google_drive.ps1 en Windows.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
    missing = [r['path'] for r in manifest['files'] if not (artifacts / r['path']).is_file()]
    if missing: raise FileNotFoundError('Bundle incompleto en Drive:\n' + '\n'.join(missing))
    root = Path('/content') / PROJECT_NAME
    if not (root / '.git').is_dir():
        if root.exists(): raise RuntimeError(f'Ruta no administrada ya existente: {root}')
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(root)], check=True)
    current = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    if current != GIT_COMMIT:
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', GIT_COMMIT], check=True)
    subprocess.run(['git', '-C', str(root), 'sparse-checkout', 'set', 'scripts_auxiliares'], check=True)
    subprocess.run(['git', '-C', str(root), 'checkout', '--detach', GIT_COMMIT], check=True)
    for name in ('datos', 'modelos', 'resultados'):
        local, target = root / name, artifacts / name
        target.mkdir(parents=True, exist_ok=True)
        if local.is_symlink() and local.resolve() == target.resolve(): continue
        if local.is_symlink(): local.unlink()
        elif local.exists(): raise RuntimeError(f'Ruta de artefactos no administrada: {local}')
        local.symlink_to(target, target_is_directory=True)
    return root, True, artifacts, GIT_COMMIT

ROOT, IN_COLAB, ARTIFACT_ROOT, CODE_VERSION = _bootstrap_04_20x()
if IN_COLAB:
    packages = {'transformers': 'transformers>=4.51,<6', 'sklearn': 'scikit-learn>=1.4'}
    if NEEDS_PEFT: packages['peft'] = 'peft>=0.15,<1'
    missing_packages = [p for m, p in packages.items() if find_spec(m) is None]
    if missing_packages: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=True)
os.environ['PLN_PROJECT_ROOT'], os.environ['PLN_ARTIFACT_ROOT'] = str(ROOT), str(ARTIFACT_ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from scripts_auxiliares import experimentos_jerarquicos_4 as h4

print('Entorno:', 'Google Colab híbrido' if IN_COLAB else 'local')
print('Código:', ROOT, CODE_VERSION)
print('Artefactos persistentes:', ARTIFACT_ROOT)
print('Dispositivo:', h4.device())
print('Objetivos:', h4.TARGET_LABELS)

## 1. Datos congelados e inicialización

El experimento conserva exactamente los train/validation/test de `04_2`. El hash del dataset, manifiesto, checkpoint plano y scores se valida antes de entrenar. La referencia y el inicio se eligen por PR-AUC macro de cuatro daños en validation; test no interviene.

In [ ]:
context = h4.load_frozen_context()
display(h4.context_summary(context))
display(h4.warm_start_plan(context))

counts = pd.DataFrame({
    split: h4.four_targets(frame).sum(axis=0).astype(int)
    for split, frame in context['frames'].items()
}, index=h4.TARGET_LABELS)
display(counts)

In [ ]:
ax = counts.T.plot.bar(figsize=(11, 5), width=0.8)
ax.set_title('Positivos de las cuatro categorías por partición')
ax.set_ylabel('Chunks positivos')
ax.set_xlabel('Partición')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Diseño multitarea y transferencia

El modelo optimiza simultáneamente: pérdida binaria para cualquier daño, pérdida multietiqueta para cuatro daños y una penalización de consistencia cuando una categoría supera a la probabilidad global de daño. Las ponderaciones positivas usan la raíz de `negativos/positivos`, evitando aplicar el cociente completo bajo desbalance. La cabeza binaria ve todo `SEGURO` permitido por el aislamiento de videos; la pérdida de categorías queda enmascarada en esos negativos adicionales y continúa usando sólo el train 4:1.

El encoder parte del E5 ya afinado en `04_2`. Las filas de racismo, género y sexual se copian; `ACOSO_AMENAZA` se inicializa con el promedio de las antiguas filas de acoso y amenaza. La cabeza binaria se inicializa aleatoriamente. Todas estas piezas se actualizan durante el nuevo entrenamiento. Si más adelante existe un checkpoint terminado de `04_5` con el mismo dataset y encoder, se reutilizarán además su encoder y cabeza binaria, adaptando la cabeza de categorías.

La referencia plana combina las antiguas probabilidades como `max(p_acoso, p_amenaza)` y recalibra cuatro umbrales en validation. El máximo evita suponer independencia entre las dos salidas. La evaluación final usa bootstrap pareado por videos sobre el mismo test.

In [ ]:
WARM_START = True
FORCE = False
BOOTSTRAP_REPLICATES = 1_000

print({
    'warm_start': WARM_START,
    'force': FORCE,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
})

## 3. Reentrenamiento conjunto

La celda presenta avance por época, inferencia y bootstrap. Guarda el mejor checkpoint según PR-AUC macro de daño en validation, el historial por época, scores, umbrales, reportes, figuras, comparación y un informe Markdown. Con `FORCE=False`, un resultado compatible ya concluido sólo se carga.

In [ ]:
result = h4.run_joint_experiment(
    force=FORCE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
    warm_start=WARM_START,
)
display({
    'experimento': result['experiment_label'],
    'inicializacion': result['training']['initialization'],
    'datos_entrenamiento': result['training']['training_data'],
    'mejor_epoca': result['training']['best_epoch'],
    'decision': result['decision'],
})

In [ ]:
history = pd.DataFrame(result['training']['history'])
display(history)
ax = history.plot(
    x='epoch',
    y=['validation_binary_pr_auc', 'validation_damage_pr_auc_macro', 'validation_damage_f1_macro'],
    marker='o',
    figsize=(11, 5),
)
ax.set_title('Evolución multitarea en validation')
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Resultados, incertidumbre y uso operativo

La arquitectura sólo reemplaza la referencia plana si todo el IC 95% del delta de PR-AUC macro es positivo sin aumentar falsos negativos. También se informa una política selectiva de auto-seguro, alerta/revisión y auto-daño, calibrada exclusivamente en validation.

In [ ]:
tables = h4.load_experiment_tables(h4.JOINT_KEY)
display(tables['comparison'])
display(tables['categories'])
display(tables['bootstrap'])
display(pd.DataFrame([result['selective_operation']]))

figure_dir = h4.FIGURES_ROOT / h4.JOINT_KEY
for name in ['comparacion_global_test.png', 'recall_por_categoria_test.png', 'bootstrap_deltas_test.png']:
    display(Image(filename=str(figure_dir / name)))

In [ ]:
display(Markdown(
    f'**Resultado reproducible:** `{h4.result_path(h4.JOINT_KEY).relative_to(ROOT)}`  \n'
    f'**Informe:** `{h4.report_path(h4.JOINT_KEY).relative_to(ROOT)}`  \n'
    f'**Modelo:** `{(h4.MODEL_ROOT / h4.JOINT_KEY).relative_to(ROOT)}`'
))

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

Zhou, J., Ma, C., Long, D., Xu, G., Ding, N., Zhang, H., Xie, P., & Liu, G. (2020). Hierarchy-aware global model for hierarchical text classification. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 1106–1117). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.acl-main.104